In [5]:
import pandas as pd

# --- CONFIGURATION ---
# Replace this with your exact Excel filename if different
file_name = 'Research-Questionnaire-Responses.xlsx' 
# ---------------------

print("Loading data...")

# 1. Load Data using read_excel
# We use engine='openpyxl' which is standard for .xlsx files
try:
    df = pd.read_excel(file_name, engine='openpyxl')
except FileNotFoundError:
    print(f"ERROR: The file '{file_name}' was not found.")
    print("Please make sure the Excel file is in the same folder as this script.")
    exit()
except Exception as e:
    print(f"An error occurred while loading: {e}")
    exit()

# Clean column headers (remove accidental spaces)
df.columns = df.columns.str.strip()

# 2. Filter Eligibility
# We look for the column asking about eligibility
eligible_col = 'Do you meet all of the eligibility criteria to participate in this survey?'

if eligible_col not in df.columns:
    print(f"Warning: Column '{eligible_col}' not found. Checking first few columns...")
    # Fallback: assume it is the last column or specific index if name changed
    pass 
else:
    # Filter for 'Yes'
    df = df[df[eligible_col].astype(str).str.startswith('Yes')].copy()
    print(f"Filtered eligible participants. Count: {len(df)}")

# 3. Rename Columns
new_cols = ['Age', 'Gender', 'CivilStatus', 'Nationality', 'Education', 
            'StayLength', 'Industry', 'Tenure', 'Role', 'Income']

# Generate scale item names safely
scale_cols = (
    [f'POS_Recog_{i}' for i in range(1,6)] +
    [f'POS_Sup_{i}'   for i in range(1,6)] +
    [f'POS_Cond_{i}'  for i in range(1,6)] +
    [f'POS_Fair_{i}'  for i in range(1,6)] +
    [f'ES_Int_{i}'    for i in range(1,6)] +
    [f'ES_Ext_{i}'    for i in range(1,6)]
)

# Apply slicing (Selecting columns by position)
# Demographics are usually columns 2 to 12 (indices 2:12)
# Scale questions are usually columns 12 to 42 (indices 12:42)
try:
    df_demo = df.iloc[:, 2:12]   # 10 Demographic columns
    df_scales = df.iloc[:, 12:42] # 30 Scale columns
    
    # Combine
    df_clean = pd.concat([df_demo, df_scales], axis=1)
    
    # Assign new names
    if len(df_clean.columns) == len(new_cols) + len(scale_cols):
        df_clean.columns = new_cols + scale_cols
    else:
        print(f"Error: Column count mismatch. Selected {len(df_clean.columns)} columns, expected {len(new_cols) + len(scale_cols)}.")
        print("Please check your Excel file structure.")
        exit()

except Exception as e:
    print(f"Error during column slicing: {e}")
    exit()

# 4. Convert Text to Numbers
likert_map = {
    'strongly disagree': 1, 
    'disagree': 2, 
    'neutral': 3, 
    'agree': 4, 
    'strongly agree': 5
}

for col in scale_cols:
    # Convert to string, lowercase, strip spaces, then map
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip().map(likert_map)

# 5. Calculate Means
# Indices: 0-9 are Demographics. 10-29 are POS (20 items). 30-39 are ES (10 items).
df_clean['POS_Total_Mean'] = df_clean.iloc[:, 10:30].mean(axis=1)
df_clean['ES_Total_Mean'] = df_clean.iloc[:, 30:40].mean(axis=1)

# 6. Create Groups for ANOVA (Low/Moderate/High)
p33 = df_clean['POS_Total_Mean'].quantile(0.33)
p66 = df_clean['POS_Total_Mean'].quantile(0.66)

def classify(x):
    if pd.isna(x): return 'Unknown'
    if x <= p33: return 'Low Support'
    elif x <= p66: return 'Moderate Support'
    else: return 'High Support'

df_clean['POS_Group'] = df_clean['POS_Total_Mean'].apply(classify)

# 7. Save to CSV for Jamovi
output_file = 'Cleaned_Data_Jamovi.csv'
df_clean.to_csv(output_file, index=False)
print(f"Success! File saved as: {output_file}")

Loading data...
Filtered eligible participants. Count: 390
Success! File saved as: Cleaned_Data_Jamovi.csv


In [8]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# --- CONFIGURATION ---
# Replace this with your exact Excel filename
file_name = 'Research-Questionnaire-Responses.xlsx'
# ---------------------

print(f"Loading {file_name}...")

# 1. Load and Clean Data
try:
    # UPDATED: Uses read_excel instead of read_csv
    df = pd.read_excel(file_name, engine='openpyxl')
except FileNotFoundError:
    print(f"Error: The file '{file_name}' was not found.")
    exit()
except Exception as e:
    print(f"Error loading file: {e}")
    exit()

# Filter Eligibility
# Finds the column containing 'eligibility criteria' automatically to avoid exact name mismatch
eligible_col_candidates = [c for c in df.columns if 'eligibility criteria' in str(c).lower()]

if not eligible_col_candidates:
    print("Error: Could not find the Eligibility Criteria column.")
    exit()

eligible_col = eligible_col_candidates[0]
df_clean = df[df[eligible_col].astype(str).str.startswith('Yes')].copy()
print(f"Filtered eligible participants: {len(df_clean)}")

# Rename Columns
# Assumes standard structure based on your previous file:
# 0-11: Metadata & Demographics
# 12-31: POS Items (20 items)
# 32-41: ES Items (10 items)

# Safety check for column count
if len(df_clean.columns) < 42:
    print("Warning: Dataset has fewer columns than expected. Check the Excel file format.")

pos_cols = df_clean.columns[12:32]
es_cols = df_clean.columns[32:42]

# Map Likert Scale Text to Numbers
scale_map = {
    'strongly disagree': 1,
    'disagree': 2,
    'neutral': 3,
    'agree': 4,
    'strongly agree': 5
}

# Convert to numeric
print("Converting Likert scales...")
for col in pos_cols:
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip().map(scale_map)
for col in es_cols:
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip().map(scale_map)

# Drop rows with NaNs in scale items if any
original_len = len(df_clean)
df_clean = df_clean.dropna(subset=list(pos_cols) + list(es_cols))
if len(df_clean) < original_len:
    print(f"Dropped {original_len - len(df_clean)} rows with missing scale data.")

# Calculate Means
# POS Subscales (5 items each)
df_clean['POS_Recog'] = df_clean[pos_cols[0:5]].mean(axis=1)
df_clean['POS_Sup'] = df_clean[pos_cols[5:10]].mean(axis=1)
df_clean['POS_Cond'] = df_clean[pos_cols[10:15]].mean(axis=1)
df_clean['POS_Fair'] = df_clean[pos_cols[15:20]].mean(axis=1)
df_clean['POS_Total'] = df_clean[pos_cols].mean(axis=1)

# ES Subscales (5 items each)
df_clean['ES_Int'] = df_clean[es_cols[0:5]].mean(axis=1)
df_clean['ES_Ext'] = df_clean[es_cols[5:10]].mean(axis=1)
df_clean['ES_Total'] = df_clean[es_cols].mean(axis=1)

# Grouping for ANOVA (Low/Mod/High based on POS)
p33 = df_clean['POS_Total'].quantile(0.33)
p66 = df_clean['POS_Total'].quantile(0.66)

def classify_pos(x):
    if x <= p33: return 'Low Support'
    elif x <= p66: return 'Moderate Support'
    else: return 'High Support'

df_clean['POS_Group'] = df_clean['POS_Total'].apply(classify_pos)

# --- Analysis Functions ---

def cronbach_alpha(df_items):
    # Cronbach's alpha calculation
    k = df_items.shape[1]
    var_items = df_items.var(axis=0, ddof=1).sum()
    var_total = df_items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - (var_items / var_total))

print("\nRunning Statistical Analysis...")

# 1. Reliability
alpha_pos = cronbach_alpha(df_clean[pos_cols])
alpha_es = cronbach_alpha(df_clean[es_cols])

# 2. Descriptives (Mean, SD, Interpretation)
desc_cols = ['POS_Recog', 'POS_Sup', 'POS_Cond', 'POS_Fair', 'POS_Total', 
             'ES_Int', 'ES_Ext', 'ES_Total']
descriptives = df_clean[desc_cols].describe().T[['mean', 'std', 'min', 'max']]

# 3. Correlation (Pearson)
corr_res = stats.pearsonr(df_clean['POS_Total'], df_clean['ES_Total'])

# 4. ANOVA
# IV: POS_Group, DV: ES_Total
groups = [df_clean[df_clean['POS_Group'] == g]['ES_Total'] for g in ['Low Support', 'Moderate Support', 'High Support']]
anova_res = stats.f_oneway(*groups)

# Post-hoc (Tukey)
tukey = pairwise_tukeyhsd(endog=df_clean['ES_Total'], groups=df_clean['POS_Group'], alpha=0.05)

# Print Results
print("\n" + "="*30)
print("       JAMOVI ANALYSIS RESULTS")
print("="*30)

print("\n--- 1. RELIABILITY (Cronbach's Alpha) ---")
print(f"POS Alpha: {alpha_pos:.3f} (Target > 0.7)")
print(f"ES Alpha:  {alpha_es:.3f} (Target > 0.7)")

print("\n--- 2. DESCRIPTIVES ---")
print(descriptives.round(3))

print("\n--- 3. CORRELATION (POS vs ES) ---")
print(f"Pearson r = {corr_res[0]:.3f}")
print(f"p-value   = {corr_res[1]:.4f}")
if corr_res[1] < 0.05:
    print(">> Significant Relationship FOUND")
else:
    print(">> No Significant Relationship")

print("\n--- 4. ANOVA (Does Support Level affect Success?) ---")
print(f"F-statistic = {anova_res.statistic:.3f}")
print(f"p-value     = {anova_res.pvalue:.4f}")

if anova_res.pvalue < 0.05:
    print(">> Significant Difference FOUND between groups")
    print("\n--- POST-HOC TEST (Tukey HSD) ---")
    print(tukey)
else:
    print(">> No significant difference between groups")

Loading Research-Questionnaire-Responses.xlsx...
Filtered eligible participants: 390
Converting Likert scales...

Running Statistical Analysis...

       JAMOVI ANALYSIS RESULTS

--- 1. RELIABILITY (Cronbach's Alpha) ---
POS Alpha: 0.964 (Target > 0.7)
ES Alpha:  0.937 (Target > 0.7)

--- 2. DESCRIPTIVES ---
            mean    std   min  max
POS_Recog  3.623  0.795  1.00  5.0
POS_Sup    3.772  0.734  1.00  5.0
POS_Cond   3.751  0.704  1.00  5.0
POS_Fair   3.694  0.735  1.00  5.0
POS_Total  3.710  0.676  1.05  5.0
ES_Int     3.789  0.737  1.00  5.0
ES_Ext     3.611  0.730  1.00  5.0
ES_Total   3.700  0.691  1.00  5.0

--- 3. CORRELATION (POS vs ES) ---
Pearson r = 0.832
p-value   = 0.0000
>> Significant Relationship FOUND

--- 4. ANOVA (Does Support Level affect Success?) ---
F-statistic = 210.592
p-value     = 0.0000
>> Significant Difference FOUND between groups

--- POST-HOC TEST (Tukey HSD) ---
        Multiple Comparison of Means - Tukey HSD, FWER=0.05        
   group1         gr

In [1]:
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURATION
# ==========================================
INPUT_FILE = 'Research-Questionnaire-Responses.xlsx'
OUTPUT_FILE = 'Cleaned_Data_Jamovi_385.csv'
TARGET_SAMPLE_SIZE = 385

# Likert Scale Mapping (Case Insensitive)
LIKERT_MAP = {
    'strongly disagree': 1,
    'disagree': 2,
    'neutral': 3,
    'agree': 4,
    'strongly agree': 5
}

# ==========================================
# 1. LOAD AND INSPECT
# ==========================================
print(f"Loading data from '{INPUT_FILE}'...")
try:
    df = pd.read_excel(INPUT_FILE, engine='openpyxl')
except Exception as e:
    print(f"CRITICAL ERROR: Could not load file. {e}")
    exit()

# Clean raw headers
df.columns = df.columns.str.strip()

# ==========================================
# 2. ELIGIBILITY FILTERING
# ==========================================
# Identify the eligibility column (looks for keyword 'eligibility')
eligibility_col = next((col for col in df.columns if 'eligibility' in str(col).lower()), None)

if eligibility_col:
    initial_count = len(df)
    # Keep only those starting with 'Yes'
    df = df[df[eligibility_col].astype(str).str.strip().str.startswith('Yes', na=False)]
    print(f"Eligibility Filter: Reduced from {initial_count} to {len(df)} participants.")
else:
    print("WARNING: Eligibility column not found. Skipping filter.")

# ==========================================
# 3. STATISTICAL PRUNING (TARGET N=385)
# ==========================================
# We use random_state=42 for reproducibility
if len(df) > TARGET_SAMPLE_SIZE:
    print(f"Pruning dataset to exactly {TARGET_SAMPLE_SIZE} participants...")
    df = df.sample(n=TARGET_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
else:
    print(f"Pruning not required. Count ({len(df)}) is <= {TARGET_SAMPLE_SIZE}.")

# ==========================================
# 4. COLUMN MAPPING & RENAMING
# ==========================================
# This order matches Table 2 in your Questionnaire Mapping PDF exactly.

scale_columns_ordered = (
    # H1: Contribution Recognition (REC1 - REC5)
    [f'POS_Recog_{i}' for i in range(1, 6)] +
    
    # H2: Supervisor Support (SUP1 - SUP5)
    [f'POS_Sup_{i}' for i in range(1, 6)] +
    
    # H3: Job Conditions (JOB1 - JOB5)
    [f'POS_Cond_{i}' for i in range(1, 6)] +
    
    # H4: Fair Treatment (FAIR1 - FAIR5)
    [f'POS_Fair_{i}' for i in range(1, 6)] +
    
    # H5: Intrinsic Success (INT1 - INT5)
    [f'ES_Int_{i}' for i in range(1, 6)] +
    
    # H6: Extrinsic Success (EXT1 - EXT5)
    [f'ES_Ext_{i}' for i in range(1, 6)]
)

try:
    # Select Demographic Data (Columns 3 to 13, assuming 0-2 are Metadata)
    # Matches DEM1 (Age) through DEM10 (Income)
    df_demos = df.iloc[:, 3:13].copy() 
    df_demos.columns = ['Age', 'Gender', 'CivilStatus', 'Nationality', 'Education', 
                        'StayLength', 'Industry', 'Tenure', 'Role', 'Income']

    # Select Scale Data (Columns 13 to 43)
    # Matches REC1 through EXT5 (30 items total)
    df_scales = df.iloc[:, 13:43].copy()
    
    if len(df_scales.columns) != 30:
        raise ValueError(f"Expected 30 scale columns, found {len(df_scales.columns)}")
        
    df_scales.columns = scale_columns_ordered

    # Combine
    df_clean = pd.concat([df_demos, df_scales], axis=1)
    
except Exception as e:
    print(f"Mapping Error: {e}")
    print("Check if the column indices (3:13 and 13:43) match your Excel file layout.")
    exit()

# ==========================================
# 5. RECODING & COMPUTING
# ==========================================
print("Recoding Likert scales and computing means...")
for col in scale_columns_ordered:
    df_clean[col] = df_clean[col].astype(str).str.lower().str.strip().map(LIKERT_MAP)

# Compute Sub-scales (Matching H1-H6)
df_clean['POS_Recog_Mean'] = df_clean[[c for c in scale_columns_ordered if 'POS_Recog' in c]].mean(axis=1)
df_clean['POS_Sup_Mean']   = df_clean[[c for c in scale_columns_ordered if 'POS_Sup' in c]].mean(axis=1)
df_clean['POS_Cond_Mean']  = df_clean[[c for c in scale_columns_ordered if 'POS_Cond' in c]].mean(axis=1)
df_clean['POS_Fair_Mean']  = df_clean[[c for c in scale_columns_ordered if 'POS_Fair' in c]].mean(axis=1)
df_clean['ES_Int_Mean']    = df_clean[[c for c in scale_columns_ordered if 'ES_Int' in c]].mean(axis=1)
df_clean['ES_Ext_Mean']    = df_clean[[c for c in scale_columns_ordered if 'ES_Ext' in c]].mean(axis=1)

# Compute Total Composites
df_clean['POS_Total_Mean'] = df_clean[[c for c in scale_columns_ordered if 'POS' in c]].mean(axis=1)
df_clean['ES_Total_Mean'] = df_clean[[c for c in scale_columns_ordered if 'ES' in c]].mean(axis=1)

# ==========================================
# 6. GROUPING (LOW/MOD/HIGH)
# ==========================================
# Percentiles calculated on the pruned N=385 dataset
p33 = df_clean['POS_Total_Mean'].quantile(0.33)
p66 = df_clean['POS_Total_Mean'].quantile(0.66)

def categorize_support(score):
    if pd.isna(score): return np.nan
    if score <= p33: return 'Low Support'
    elif score <= p66: return 'Moderate Support'
    else: return 'High Support'

df_clean['POS_Group'] = df_clean['POS_Total_Mean'].apply(categorize_support)

# Export
df_clean.to_csv(OUTPUT_FILE, index=False)
print(f"SUCCESS: Processed data saved to '{OUTPUT_FILE}' (N={len(df_clean)})")

Loading data from 'Research-Questionnaire-Responses.xlsx'...
Eligibility Filter: Reduced from 414 to 390 participants.
Pruning dataset to exactly 385 participants...
Recoding Likert scales and computing means...
SUCCESS: Processed data saved to 'Cleaned_Data_Jamovi_385.csv' (N=385)
